# Final Experiments for DAMK-Net / MK-MNet

Notebook nay de chay cac thuc nghiem bo sung tren Google Colab voi structure repo moi.

Muc tieu:
- multi-seed augmentation comparison
- validation-first threshold selection
- Grad-CAM cho operating point chinh
- optional lightweight profiling

Notebook co sanity checks de tranh loi path, checkpoint, dataset, va output folder.

## Cach dung nhanh

1. Dat repo `lightweight-medical-model` trong Google Drive theo dung `PROJECT_DIR` ben duoi, hoac sua lai path cho khop.
2. Dam bao dataset BUSI va checkpoint `.pt` ton tai trong mot trong cac `ARTIFACT_CANDIDATES`.
3. Chay notebook tu tren xuong duoi.

Notebook nay theo cung convention path voi cac notebook Colab cu trong repo de tranh loi vo van ve duong dan.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Toggle tung nhom experiment
RUN_AUGMENTATION = True
RUN_THRESHOLD_PROTOCOL = True
RUN_GRADCAM = True
RUN_PROFILING = False

# Giong convention cua cac notebook Colab cu trong repo
DRIVE_ROOT = Path('/content/drive/MyDrive/Master/LTAINC')
PROJECT_DIR = DRIVE_ROOT / 'lightweight-medical-model'

# Dataset/checkpoint co the nam ngay trong repo hoac nam o mot artifact root khac tren Drive
ARTIFACT_CANDIDATES = [
    PROJECT_DIR,
    DRIVE_ROOT / 'colab-result/lightweight-medical-model',
    PROJECT_DIR / 'colab-result/lightweight-medical-model',
]

DATASET_RELATIVE = Path('data/busi')
CHECKPOINT_RELATIVE = Path('outputs/busi/mk_mnet/img_224/width_0.5/oversample_1/lambda_0.8/lr_0.0003/weight_decay_0.0001/patience_10/best_model.pt')

AUG_OUTPUT_ROOT = PROJECT_DIR / 'artifacts' / 'augmentation-comparison'
THRESHOLD_OUTPUT_ROOT = PROJECT_DIR / 'artifacts' / 'threshold-validation-protocol' / 'mk_mnet_width05_os'
GRADCAM_OUTPUT_ROOT = PROJECT_DIR / 'artifacts' / 'gradcam'
PROFILE_OUTPUT_CSV = PROJECT_DIR / 'artifacts' / 'model-profiles' / 'profile.csv'

AUG_POLICIES = ['none', 'rotate', 'translate_y', 'scale', 'horizontal_flip']
AUG_SEEDS = [42, 123, 2026]

MODEL_NAME = 'mk_mnet'
WIDTH_MULT = 0.5
PROFILE_MODELS = ['mednet', 'mk_mnet', 'r_cbam_mnet']
PROFILE_WIDTH_MULTS = [0.25, 0.5, 1.0]
PROFILE_ITERATIONS = 100


In [ ]:
import os
import shutil
import subprocess
import sys


def resolve_artifact_root(candidates, required_relative_path: Path) -> Path:
    for root in candidates:
        if (root / required_relative_path).exists():
            return root
    checked = "\n".join(f"- {root / required_relative_path}" for root in candidates)
    raise FileNotFoundError(
        f"Khong tim thay artifact can thiet. Da kiem tra:\n{checked}"
    )


def run(cmd, cwd=None):
    print("\n>>>", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)


def ensure_uv():
    if shutil.which('uv'):
        print('uv da san sang')
        return
    run([sys.executable, '-m', 'pip', 'install', '-U', 'uv'])


def print_tree(root: Path, max_depth=2):
    root = root.resolve()
    print(f"\nTree: {root}")
    for path in sorted(root.rglob('*')):
        depth = len(path.relative_to(root).parts)
        if depth > max_depth:
            continue
        prefix = '  ' * (depth - 1)
        suffix = '/' if path.is_dir() else ''
        print(f"{prefix}{path.name}{suffix}")


In [ ]:
assert PROJECT_DIR.is_dir(), f"Khong tim thay repo code: {PROJECT_DIR}"
assert (PROJECT_DIR / 'pyproject.toml').is_file(), 'Thieu pyproject.toml'
assert (PROJECT_DIR / 'scripts/experiments/run_augmentation_comparison.py').is_file(), 'Thieu experiment runner'

ARTIFACT_ROOT = resolve_artifact_root(ARTIFACT_CANDIDATES, CHECKPOINT_RELATIVE)
DATASET_ROOT = resolve_artifact_root(ARTIFACT_CANDIDATES, DATASET_RELATIVE)
DATASET_DIR = DATASET_ROOT / DATASET_RELATIVE
CHECKPOINT_PATH = ARTIFACT_ROOT / CHECKPOINT_RELATIVE

print("Code dir:      ", PROJECT_DIR)
print("Artifact root: ", ARTIFACT_ROOT)
print("Dataset root:  ", DATASET_ROOT)
print("Dataset dir:   ", DATASET_DIR)
print("Checkpoint:    ", CHECKPOINT_PATH)
print("Aug output:    ", AUG_OUTPUT_ROOT)
print("Threshold out: ", THRESHOLD_OUTPUT_ROOT)
print("Grad-CAM out:  ", GRADCAM_OUTPUT_ROOT)
print("Profile csv:   ", PROFILE_OUTPUT_CSV)


In [ ]:
import os

%cd {PROJECT_DIR}
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
print('PYTHONPATH:', os.environ['PYTHONPATH'])
# Bo comment neu can dong bo code truoc khi chay.
# !git pull origin main


In [ ]:
assert DATASET_DIR.is_dir(), f"Dataset dir khong ton tai: {DATASET_DIR}"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint khong ton tai: {CHECKPOINT_PATH}"
print('Sanity checks OK')
print_tree(PROJECT_DIR / 'scripts', max_depth=2)
ensure_uv()
run(['uv', 'sync'], cwd=PROJECT_DIR)


## 1. Multi-seed augmentation comparison

In [ ]:
if RUN_AUGMENTATION:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.experiments.run_augmentation_comparison',
        '--dataset-dir', str(DATASET_DIR),
        '--policies', *AUG_POLICIES,
        '--seeds', *[str(seed) for seed in AUG_SEEDS],
        '--resume',
    ]
    run(cmd, cwd=PROJECT_DIR)
else:
    print('Bo qua augmentation comparison')


In [ ]:
import pandas as pd

summary_csv = AUG_OUTPUT_ROOT / 'comparison_summary.csv'
status_csv = AUG_OUTPUT_ROOT / 'comparison_status.csv'

if summary_csv.exists():
    display(pd.read_csv(summary_csv))
else:
    print('Chua co comparison_summary.csv')

if status_csv.exists():
    display(pd.read_csv(status_csv))
else:
    print('Chua co comparison_status.csv')


## 2. Validation-first threshold selection

In [ ]:
if RUN_THRESHOLD_PROTOCOL:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.experiments.run_threshold_validation_protocol',
        '--model', MODEL_NAME,
        '--width-mult', str(WIDTH_MULT),
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--output-root', str(THRESHOLD_OUTPUT_ROOT),
        '--resume',
    ]
    run(cmd, cwd=PROJECT_DIR)
else:
    print('Bo qua threshold validation protocol')


In [ ]:
selected_summary = THRESHOLD_OUTPUT_ROOT / 'selected_threshold_summary.csv'
val_sweep = THRESHOLD_OUTPUT_ROOT / 'val' / 'threshold_sensitivity.csv'
test_eval = THRESHOLD_OUTPUT_ROOT / 'test' / 'threshold_sensitivity.csv'

for path in [selected_summary, val_sweep, test_eval]:
    print('\n', path)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print('Chua co file')


## 3. Grad-CAM for the main operating point

In [ ]:
if RUN_GRADCAM:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.visualization.generate_gradcam',
        '--model', MODEL_NAME,
        '--width-mult', str(WIDTH_MULT),
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--outputs-root', str(GRADCAM_OUTPUT_ROOT),
        '--all-samples',
        '--target-class', 'true',
    ]
    run(cmd, cwd=PROJECT_DIR)
else:
    print('Bo qua Grad-CAM')


In [ ]:
from IPython.display import Image, display

gradcam_candidates = list(GRADCAM_OUTPUT_ROOT.rglob('*.png'))
print(f'Tim thay {len(gradcam_candidates)} file PNG trong {GRADCAM_OUTPUT_ROOT}')

for image_path in gradcam_candidates[:8]:
    print(image_path)
    display(Image(filename=str(image_path), width=480))


## 4. Optional profiling

In [ ]:
if RUN_PROFILING:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.analysis.profile_models',
        '--models', *PROFILE_MODELS,
        '--width-mults', *[str(w) for w in PROFILE_WIDTH_MULTS],
        '--iterations', str(PROFILE_ITERATIONS),
        '--output', str(PROFILE_OUTPUT_CSV),
    ]
    run(cmd, cwd=PROJECT_DIR)
else:
    print('Bo qua profiling')


In [ ]:
if PROFILE_OUTPUT_CSV.exists():
    display(pd.read_csv(PROFILE_OUTPUT_CSV))
else:
    print('Chua co profile.csv')


## Output de dua vao report

- `artifacts/augmentation-comparison/comparison_summary.csv`: bang mean/std cho augmentation.
- `artifacts/threshold-validation-protocol/.../selected_threshold_summary.csv`: chon tau tren validation va metric test sau khi khoa tau.
- `artifacts/gradcam/.../*.png`: hinh dinh tinh cho operating point chinh.
- `artifacts/model-profiles/profile.csv`: params, FLOPs, latency local.

Sau khi chay xong, download cac CSV va PNG can thiet tu Drive de cap nhat lai report.